In [11]:
import os
import re
import time
import random
import unicodedata
from urllib.parse import urlparse, urlunparse

import openpyxl
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# KONFIGURATION
# ============================================================

INPUT_FILE = (
    "/Users/Familie/Documents/Uni_Regensburg/"
    "A_Digital_Humanities/2SE_SS26/"
    "Natural_Language_Engineering/projekt/"
    "frame_detect_NLP/Y_test_additional_dataset/datasets/"
    "bearbeitet/chinese_dataset/"
    "ground_truth_chinese_extension_semeval.xlsx"
)

OUTPUT_FILE = (
    "/Users/Familie/Documents/Uni_Regensburg/"
    "A_Digital_Humanities/2SE_SS26/"
    "Natural_Language_Engineering/projekt/"
    "frame_detect_NLP/Y_test_additional_dataset/datasets/"
    "bearbeitet/chinese_dataset/"
    "ground_truth_chinese_extension_semeval_with_titles.xlsx"
)

LOG_FILE = (
    "/Users/Familie/Documents/Uni_Regensburg/"
    "A_Digital_Humanities/2SE_SS26/"
    "Natural_Language_Engineering/projekt/"
    "frame_detect_NLP/Y_test_additional_dataset/datasets/"
    "bearbeitet/chinese_dataset/"
    "headline_crawl_log.txt"
)

URL_COLUMN = "Filename"
TITLE_COLUMN = "Headline_GER"

# Zwischen Requests warten.
MIN_DELAY = 0.8
MAX_DELAY = 1.8

REQUEST_TIMEOUT = 20

# Wenn True:
# Bereits vorhandene Headline_GER-Werte werden nicht erneut gecrawlt.
SKIP_EXISTING = True

# Wenn True:
# Nach einem fehlgeschlagenen Crawl wird "err" geschrieben.
WRITE_ERR = True


# ============================================================
# USER-AGENT
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": (
        "zh-CN,zh;q=0.9,en-US;q=0.7,en;q=0.6"
    ),
    "Connection": "keep-alive",
}


# ============================================================
# DOMAINEN
# ============================================================

NTDTV = {
    "www.ntdtv.com",
}

EPOCH_TIMES = {
    "www.epochtimes.com",
}

RFI = {
    "www.rfi.fr",
}

VOA = {
    "www.voachinese.com",
}

REUTERS = {
    "cn.reuters.com",
}

THE_PAPER = {
    "www.thepaper.cn",
}

XINHUA = {
    "www.news.cn",
    "hlj.news.cn",
    "bj.news.cn",
}

DW = {
    "www.dw.com",
}

CHINA_DAILY = {
    "cn.chinadaily.com.cn",
    "china.chinadaily.com.cn",
    "finance.chinadaily.com.cn",
    "chuangxin.chinadaily.com.cn",
    "caijing.chinadaily.com.cn",
    "hlj.chinadaily.com.cn",
}

NYT = {
    "cn.nytimes.com",
}

FT = {
    "cn.ft.com",
}

BBC = {
    "www.bbc.com",
}

CHINA_DIGITAL_TIMES = {
    "chinadigitaltimes.net",
}

RFA = {
    "www.rfa.org",
}

SPECIAL_OTHER = {
    "www.gsftyg.com",
}


# ============================================================
# SESSION MIT RETRIES
# ============================================================

def create_session():
    session = requests.Session()

    retry_strategy = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy,
        pool_connections=10,
        pool_maxsize=10,
    )

    session.mount("http://", adapter)
    session.mount("https://", adapter)

    session.headers.update(HEADERS)

    return session


# ============================================================
# URL NORMALISIERUNG
# ============================================================

def normalize_url(url):
    """
    Normalisiert problematische URLs.

    Insbesondere:
    http://www.rfi.fr//cn/...
        ->
    https://www.rfi.fr/cn/...
    """

    if not isinstance(url, str):
        return None

    url = url.strip()

    if not url:
        return None

    if not re.match(r"^https?://", url, re.IGNORECASE):
        return None

    parsed = urlparse(url)

    scheme = parsed.scheme.lower()
    hostname = parsed.netloc.lower()

    # RFI:
    # Doppelten Slash im Pfad entfernen.
    path = re.sub(r"/{2,}", "/", parsed.path)

    # Bei RFI HTTPS bevorzugen.
    if hostname == "www.rfi.fr":
        scheme = "https"

    normalized = urlunparse(
        (
            scheme,
            hostname,
            path,
            parsed.params,
            parsed.query,
            parsed.fragment,
        )
    )

    return normalized


# ============================================================
# TEXT-BEREINIGUNG
# ============================================================

def clean_title(text):
    """
    Bereinigt einen extrahierten Titel.
    """

    if text is None:
        return None

    text = str(text)

    # Unicode normalisieren
    text = unicodedata.normalize("NFKC", text)

    # Whitespace vereinheitlichen
    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    if not text:
        return None

    # Häufige Trennzeichen am Anfang/Ende entfernen
    text = text.strip(" |-_—–·•")

    if not text:
        return None

    return text


# ============================================================
# CHINESISCH-ERKENNUNG
# ============================================================

def count_chinese_characters(text):
    if not text:
        return 0

    count = 0

    for char in text:
        code = ord(char)

        if (
            0x3400 <= code <= 0x4DBF
            or 0x4E00 <= code <= 0x9FFF
            or 0xF900 <= code <= 0xFAFF
        ):
            count += 1

    return count


def contains_chinese(text):
    return count_chinese_characters(text) >= 2


# ============================================================
# OFFENSICHTLICH UNGEEIGNETE TITEL
# ============================================================

GENERIC_BAD_TITLES = {
    "首页",
    "首頁",
    "新闻",
    "新聞",
    "国际",
    "國際",
    "中国",
    "中國",
    "财经",
    "財經",
    "视频",
    "視頻",
    "图片",
    "圖片",
    "搜索",
    "搜尋",
    "登录",
    "登入",
    "注册",
    "註冊",
    "404",
    "page not found",
    "error",
}


def looks_like_navigation_or_site_name(title):
    if not title:
        return True

    normalized = title.lower().strip()

    if normalized in {
        x.lower() for x in GENERIC_BAD_TITLES
    }:
        return True

    # Typische Fehlerseiten
    error_patterns = [
        "404",
        "page not found",
        "access denied",
        "forbidden",
        "error",
        "robot check",
        "captcha",
    ]

    if any(pattern in normalized for pattern in error_patterns):
        return True

    return False


# ============================================================
# TITEL-PLAUSIBILITÄT
# ============================================================

def title_is_plausible(title, source_domain=None):
    """
    Konservative Prüfung.

    Wir wollen lieber "err" als einen falschen Titel.
    """

    title = clean_title(title)

    if not title:
        return False

    if looks_like_navigation_or_site_name(title):
        return False

    chinese_count = count_chinese_characters(title)

    # Da wir chinesische Originaltitel erwarten,
    # muss der Titel mindestens zwei chinesische Zeichen enthalten.
    if chinese_count < 2:
        return False

    # Extrem lange Inhalte sind wahrscheinlich kein Titel,
    # sondern Volltext / Beschreibung.
    if len(title) > 300:
        return False

    # Ein einzelnes sehr langes Segment ohne Satzzeichen
    # ist ebenfalls verdächtig.
    if len(title) > 220 and chinese_count < 20:
        return False

    # Meta-/UI-Fragmente
    bad_fragments = [
        "cookie",
        "javascript",
        "subscribe",
        "newsletter",
        "copyright",
        "privacy policy",
        "隐私政策",
        "隐私權",
        "登录",
        "登入",
        "注册",
        "註冊",
    ]

    lower_title = title.lower()

    for fragment in bad_fragments:
        if fragment.lower() in lower_title:
            return False

    return True


# ============================================================
# DUPLIKATE AUS TITEL-KANDIDATEN ENTFERNEN
# ============================================================

def unique_candidates(candidates):
    result = []
    seen = set()

    for candidate in candidates:
        candidate = clean_title(candidate)

        if not candidate:
            continue

        key = candidate.casefold()

        if key in seen:
            continue

        seen.add(key)
        result.append(candidate)

    return result


# ============================================================
# META-TITEL
# ============================================================

def extract_meta_candidates(soup):
    candidates = []

    # OpenGraph
    for tag in soup.find_all("meta"):
        prop = tag.get("property", "")
        name = tag.get("name", "")
        content = tag.get("content")

        if not content:
            continue

        prop = prop.lower().strip()
        name = name.lower().strip()

        if prop in {
            "og:title",
            "twitter:title",
        }:
            candidates.append(content)

        if name in {
            "title",
            "headline",
        }:
            candidates.append(content)

    return candidates


# ============================================================
# JSON-LD
# ============================================================

def extract_jsonld_candidates(soup):
    candidates = []

    scripts = soup.find_all(
        "script",
        type=re.compile(
            r"application/ld\+json",
            re.IGNORECASE,
        ),
    )

    import json

    def process_object(obj):
        if isinstance(obj, dict):

            # Häufige Schema.org-Felder
            for key in [
                "headline",
                "name",
                "alternativeHeadline",
            ]:
                value = obj.get(key)

                if isinstance(value, str):
                    candidates.append(value)

            # @graph
            graph = obj.get("@graph")

            if isinstance(graph, list):
                for item in graph:
                    process_object(item)

        elif isinstance(obj, list):
            for item in obj:
                process_object(item)

    for script in scripts:

        raw = script.string or script.get_text()

        if not raw:
            continue

        raw = raw.strip()

        try:
            data = json.loads(raw)
            process_object(data)

        except Exception:
            continue

    return candidates


# ============================================================
# ALLGEMEINE HTML-KANDIDATEN
# ============================================================

def extract_generic_candidates(soup):
    candidates = []

    # h1 ist meist die wichtigste Quelle.
    for h1 in soup.find_all("h1"):
        text = h1.get_text(" ", strip=True)

        if text:
            candidates.append(text)

    # h2 nur als schwächerer Fallback.
    for h2 in soup.find_all("h2"):
        text = h2.get_text(" ", strip=True)

        if text:
            candidates.append(text)

    return candidates


# ============================================================
# DOMAIN-SPEZIFISCHE EXTRAKTOREN
# ============================================================

def extract_ntdtv(soup):
    candidates = []

    # NTDTV
    selectors = [
        "h1",
        "article h1",
        "[class*='article'] h1",
        "[class*='news'] h1",
        "[class*='title'] h1",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_voa(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[class*='headline'] h1",
        "[class*='title'] h1",
        "[data-testid*='headline']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_rfi(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[class*='title'] h1",
        "[class*='headline'] h1",
        "[class*='article'] h1",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_thepaper(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        ".news_txt h1",
        ".newsdetail h1",
        "[class*='news'] h1",
        "[class*='title'] h1",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_xinhua(soup):
    candidates = []

    selectors = [
        "h1",
        ".title",
        ".main-title",
        ".article-title",
        "article h1",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            text = element.get_text(" ", strip=True)

            if text:
                candidates.append(text)

    return candidates


def extract_chinadaily(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        ".article-title",
        ".title",
        "[class*='article'] h1",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_epochtimes(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        ".article-title",
        ".title",
        "[class*='article'] h1",
        "[class*='title'] h1",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_dw(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[class*='headline']",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_nyt(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[data-testid='headline']",
        "[class*='headline']",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_ft(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[class*='headline']",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_bbc(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[data-testid='headline']",
        "[class*='headline']",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_cdt(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        ".entry-title",
        ".post-title",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


def extract_rfa(soup):
    candidates = []

    selectors = [
        "h1",
        "article h1",
        "[class*='headline']",
        "[class*='title']",
    ]

    for selector in selectors:
        for element in soup.select(selector):
            candidates.append(
                element.get_text(" ", strip=True)
            )

    return candidates


# ============================================================
# DOMAIN-EXTRAKTOR AUSWÄHLEN
# ============================================================

def get_domain_specific_candidates(soup, hostname):
    if hostname in NTDTV:
        return extract_ntdtv(soup)

    if hostname in EPOCH_TIMES:
        return extract_epochtimes(soup)

    if hostname in RFI:
        return extract_rfi(soup)

    if hostname in VOA:
        return extract_voa(soup)

    if hostname in THE_PAPER:
        return extract_thepaper(soup)

    if hostname in XINHUA:
        return extract_xinhua(soup)

    if hostname in CHINA_DAILY:
        return extract_chinadaily(soup)

    if hostname in DW:
        return extract_dw(soup)

    if hostname in NYT:
        return extract_nyt(soup)

    if hostname in FT:
        return extract_ft(soup)

    if hostname in BBC:
        return extract_bbc(soup)

    if hostname in CHINA_DIGITAL_TIMES:
        return extract_cdt(soup)

    if hostname in RFA:
        return extract_rfa(soup)

    if hostname in REUTERS:
        # Reuters ist bei den alten URLs problematisch.
        # Nicht aus URL-Slug rekonstruieren.
        return []

    return []


# ============================================================
# TITEL AUS EINER SEITE EXTRAHIEREN
# ============================================================

def extract_title_from_html(html, hostname):
    soup = BeautifulSoup(html, "html.parser")

    # --------------------------------------------------------
    # 1. Domain-spezifische Kandidaten
    # --------------------------------------------------------

    domain_candidates = get_domain_specific_candidates(
        soup,
        hostname,
    )

    # --------------------------------------------------------
    # 2. JSON-LD
    # --------------------------------------------------------

    jsonld_candidates = extract_jsonld_candidates(soup)

    # --------------------------------------------------------
    # 3. OpenGraph / Meta
    # --------------------------------------------------------

    meta_candidates = extract_meta_candidates(soup)

    # --------------------------------------------------------
    # 4. Generische HTML-Kandidaten
    # --------------------------------------------------------

    generic_candidates = extract_generic_candidates(soup)

    # --------------------------------------------------------
    # Kandidaten in Prioritätsreihenfolge
    # --------------------------------------------------------

    all_candidates = []

    all_candidates.extend(domain_candidates)
    all_candidates.extend(jsonld_candidates)
    all_candidates.extend(meta_candidates)
    all_candidates.extend(generic_candidates)

    all_candidates = unique_candidates(
        all_candidates
    )

    # --------------------------------------------------------
    # Plausibilitätsprüfung
    # --------------------------------------------------------

    for candidate in all_candidates:

        candidate = clean_title(candidate)

        if title_is_plausible(
            candidate,
            hostname,
        ):
            return candidate

    return None


# ============================================================
# HTTP RESPONSE VALIDIEREN
# ============================================================

def response_looks_like_html(response):
    content_type = response.headers.get(
        "Content-Type",
        "",
    ).lower()

    if "text/html" in content_type:
        return True

    if "application/xhtml+xml" in content_type:
        return True

    # Manche ältere Webseiten setzen keinen sauberen
    # Content-Type.
    beginning = response.content[:500].lower()

    if b"<html" in beginning:
        return True

    if b"<!doctype html" in beginning:
        return True

    return False


# ============================================================
# EINZELNE URL SCRAPEN
# ============================================================

def scrape_title(url, session):
    normalized_url = normalize_url(url)

    if not normalized_url:
        return None, "invalid_url"

    parsed = urlparse(normalized_url)
    hostname = parsed.netloc.lower()

    try:
        time.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY,
            )
        )

        response = session.get(
            normalized_url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )

        status = response.status_code

        if status >= 400:
            return None, f"http_{status}"

        if not response_looks_like_html(response):
            return None, "not_html"

        # Encoding verbessern.
        if not response.encoding:
            response.encoding = response.apparent_encoding

        # Bei problematischen chinesischen Seiten:
        if response.encoding.lower() in {
            "iso-8859-1",
            "ascii",
        }:
            response.encoding = response.apparent_encoding

        title = extract_title_from_html(
            response.text,
            hostname,
        )

        if title:
            return title, "success"

        return None, "title_not_found"

    except requests.exceptions.Timeout:
        return None, "timeout"

    except requests.exceptions.TooManyRedirects:
        return None, "too_many_redirects"

    except requests.exceptions.ConnectionError:
        return None, "connection_error"

    except requests.exceptions.RequestException as exc:
        return None, (
            "request_error:"
            + exc.__class__.__name__
        )

    except Exception as exc:
        return None, (
            "unexpected_error:"
            + exc.__class__.__name__
        )


# ============================================================
# WORKBOOK LADEN
# ============================================================

def load_workbook_safely(file_path):
    """
    Absichtlich nur openpyxl.

    Dadurch umgehen wir den ursprünglichen Fehler:

        TypeError:
        Cannot convert numpy.ndarray to numpy.ndarray

    Es gibt hier keinen Pandas-/NumPy-Zwischenschritt.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Datei nicht gefunden:\n{file_path}"
        )

    workbook = openpyxl.load_workbook(
        file_path,
        data_only=False,
    )

    return workbook


# ============================================================
# HEADER FINDEN
# ============================================================

def find_header_column(worksheet, header_name):
    for cell in worksheet[1]:

        if cell.value == header_name:
            return cell.column

    return None


# ============================================================
# NEUE SPALTE ANLEGEN
# ============================================================

def ensure_title_column(worksheet):
    column = find_header_column(
        worksheet,
        TITLE_COLUMN,
    )

    if column is not None:
        return column

    new_column = worksheet.max_column + 1

    cell = worksheet.cell(
        row=1,
        column=new_column,
    )

    cell.value = TITLE_COLUMN

    return new_column


# ============================================================
# CRAWL-LOG
# ============================================================

def write_log(log_file, message):
    with open(
        log_file,
        "a",
        encoding="utf-8",
    ) as f:
        f.write(message + "\n")


# ============================================================
# HAUPTPROGRAMM
# ============================================================

def main():

    print("=" * 70)
    print("CHINESISCHE HEADLINES CRAWLER")
    print("=" * 70)

    print(f"\nEingabe:")
    print(INPUT_FILE)

    print(f"\nAusgabe:")
    print(OUTPUT_FILE)

    print(f"\nLog:")
    print(LOG_FILE)

    # --------------------------------------------------------
    # Workbook laden
    # --------------------------------------------------------

    workbook = load_workbook_safely(
        INPUT_FILE
    )

    worksheet = workbook.active

    print(
        f"\nArbeitsblatt: {worksheet.title}"
    )

    print(
        f"Zeilen: {worksheet.max_row - 1}"
    )

    print(
        f"Spalten vorher: {worksheet.max_column}"
    )

    # --------------------------------------------------------
    # URL-Spalte suchen
    # --------------------------------------------------------

    url_column = find_header_column(
        worksheet,
        URL_COLUMN,
    )

    if url_column is None:
        raise ValueError(
            f"Die Spalte '{URL_COLUMN}' "
            f"wurde nicht gefunden."
        )

    # --------------------------------------------------------
    # Headline-Spalte anlegen
    # --------------------------------------------------------

    title_column = ensure_title_column(
        worksheet
    )

    print(
        f"'{TITLE_COLUMN}' befindet sich jetzt "
        f"in Spalte {title_column}."
    )

    print(
        f"Spalten nachher: "
        f"{worksheet.max_column}"
    )

    # --------------------------------------------------------
    # Session
    # --------------------------------------------------------

    session = create_session()

    # --------------------------------------------------------
    # Statistik
    # --------------------------------------------------------

    total = worksheet.max_row - 1

    success_count = 0
    error_count = 0
    skipped_count = 0

    status_counter = {}

    # Log-Datei neu beginnen.
    with open(
        LOG_FILE,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            "HEADLINE CRAWL LOG\n"
        )

        f.write(
            "=" * 70 + "\n"
        )

        f.write(
            f"Input: {INPUT_FILE}\n"
        )

        f.write(
            f"Output: {OUTPUT_FILE}\n"
        )

        f.write(
            "=" * 70 + "\n\n"
        )

    # --------------------------------------------------------
    # Zeilen durchlaufen
    # --------------------------------------------------------

    for row_number in range(
        2,
        worksheet.max_row + 1,
    ):

        url = worksheet.cell(
            row=row_number,
            column=url_column,
        ).value

        title_cell = worksheet.cell(
            row=row_number,
            column=title_column,
        )

        # ----------------------------------------------------
        # Bereits vorhandenen Titel behalten
        # ----------------------------------------------------

        if SKIP_EXISTING:

            existing = title_cell.value

            if (
                existing is not None
                and str(existing).strip() != ""
            ):

                skipped_count += 1

                print(
                    f"[{row_number - 1}/{total}] "
                    f"SKIP: bereits vorhanden"
                )

                continue

        # ----------------------------------------------------
        # URL prüfen
        # ----------------------------------------------------

        if not isinstance(url, str):
            title_cell.value = "err"

            error_count += 1

            status_counter[
                "invalid_url"
            ] = (
                status_counter.get(
                    "invalid_url",
                    0,
                ) + 1
            )

            write_log(
                LOG_FILE,
                (
                    f"{row_number}\t"
                    f"invalid_url\t"
                    f"{url}"
                ),
            )

            continue

        # ----------------------------------------------------
        # Crawl
        # ----------------------------------------------------

        title, status = scrape_title(
            url,
            session,
        )

        status_counter[status] = (
            status_counter.get(
                status,
                0,
            ) + 1
        )

        # ----------------------------------------------------
        # Erfolg
        # ----------------------------------------------------

        if title:

            title_cell.value = title

            success_count += 1

            print(
                f"[{row_number - 1}/{total}] "
                f"OK: {title}"
            )

            write_log(
                LOG_FILE,
                (
                    f"{row_number}\t"
                    f"success\t"
                    f"{url}\t"
                    f"{title}"
                ),
            )

        # ----------------------------------------------------
        # Fehler
        # ----------------------------------------------------

        else:

            if WRITE_ERR:
                title_cell.value = "err"

            error_count += 1

            print(
                f"[{row_number - 1}/{total}] "
                f"ERR: {status}\n"
                f"    {url}"
            )

            write_log(
                LOG_FILE,
                (
                    f"{row_number}\t"
                    f"{status}\t"
                    f"{url}"
                ),
            )

        # ----------------------------------------------------
        # Zwischenspeichern
        # ----------------------------------------------------

        # Alle 25 Datensätze speichern.
        if (
            (row_number - 1) % 25 == 0
        ):

            workbook.save(
                OUTPUT_FILE
            )

            print(
                "  -> Zwischenstand gespeichert."
            )

    # --------------------------------------------------------
    # Finale Speicherung
    # --------------------------------------------------------

    workbook.save(
        OUTPUT_FILE
    )

    # --------------------------------------------------------
    # Session schließen
    # --------------------------------------------------------

    session.close()

    # --------------------------------------------------------
    # Zusammenfassung
    # --------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("CRAWLER ABGESCHLOSSEN")
    print("=" * 70)

    print(
        f"Datensätze insgesamt:     {total}"
    )

    print(
        f"Erfolgreich extrahiert:   {success_count}"
    )

    print(
        f"Fehler / 'err':           {error_count}"
    )

    print(
        f"Bereits vorhanden:        {skipped_count}"
    )

    print("\nStatus-Verteilung:")

    for status, count in sorted(
        status_counter.items(),
        key=lambda x: (-x[1], x[0]),
    ):

        print(
            f"  {status:<30} {count}"
        )

    print("\n")
    print(
        f"Excel gespeichert unter:\n"
        f"{OUTPUT_FILE}"
    )

    print(
        f"\nLog gespeichert unter:\n"
        f"{LOG_FILE}"
    )

    print("=" * 70)


# ============================================================
# START
# ============================================================

if __name__ == "__main__":
    main()

CHINESISCHE HEADLINES CRAWLER

Eingabe:
/Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_test_additional_dataset/datasets/bearbeitet/chinese_dataset/ground_truth_chinese_extension_semeval.xlsx

Ausgabe:
/Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_test_additional_dataset/datasets/bearbeitet/chinese_dataset/ground_truth_chinese_extension_semeval_with_titles.xlsx

Log:
/Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_test_additional_dataset/datasets/bearbeitet/chinese_dataset/headline_crawl_log.txt

Arbeitsblatt: Sheet1
Zeilen: 353
Spalten vorher: 16
'Headline_GER' befindet sich jetzt in Spalte 17.
Spalten nachher: 17
[1/353] OK: AI可能成为灾难 马斯克吁暂停研发加强监管
[2/353] OK: 重庆一高速隧道多车追尾 燃起熊熊大火(视频)
[3/353] OK: 美国药管机构批准紧急使用莫德纳新冠疫苗
[4/353] ERR: http_403
    http://www.rfi

    input_file = "/Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_test_additional_dataset/datasets/bearbeitet/chinese_dataset/ground_truth_chinese_extension_semeval.xlsx"
    output_file = "/Users/Familie/Documents/Uni_Regensburg/A_Digital_Humanities/2SE_SS26/Natural_Language_Engineering/projekt/frame_detect_NLP/Y_test_additional_dataset/datasets/bearbeitet/chinese_dataset/ground_truth_chinese_extention_semeval_with_titles.xlsx"
